In [1]:
import napari
import numpy as np
import imageio.v3 as iio
from pathlib import Path
import argparse
from pathlib import Path

%gui qt


In [2]:
#change depending on files to check / set input to batch in "segmented_images/DD.MM.YY"
#to do: set default and allow to give argument in python command (default to whatever)
DATA_ROOT = Path("/mnt/d/macu bilder")
BATCH_DATE = "2026-02-16"
# in readme it should explain that people go into first layer of project folder, do pwd and copy the path to set to DATA_ROOT 
#and they should also give the date_argument in bash 

BATCH_ROOT = DATA_ROOT / "choroid_segmentation/output_data/"

SEGMENTED_DIR = BATCH_ROOT / "segmented_images" / BATCH_DATE
#SEGMENTED_DIR.exists()
STATS_DIR = BATCH_ROOT / "stat_tables"

##stat_table = STATS_DIR / f"{BATCH_DATE}_output.csv"
#stat_table.exists()    

In [3]:

def load_new_case(viewer, current_image):
    global current_index
    
    viewer.layers.clear()

    current_image = iio.imread(current_image)
    #image_arr = np.asarray(current_image)
    layer1 = current_image[:,:,0]
    layer2 = current_image[:,:,2]
    layer3 = current_image[:,:,1]
    
    viewer.add_image(layer1, name = "original_image", colormap = "gray")
    viewer.add_labels(layer2, name = "vessels", opacity = 0.5)  
    viewer.add_labels(layer3, name = "region", opacity = 0.5)
    
    viewer.layers["vessels"].color = {1: "blue"}
    viewer.layers["region"].color  = {1: "orange"}
    viewer.title = 'quality control images'
    viewer.status = f"{current_index+1} / {len(image_files)}"
# conversion to numpy.arr (muss gemacht werden um die einzelnen schichten zu selecten)





In [11]:
image_files = sorted(list(SEGMENTED_DIR.glob("*.tif")))
if not image_files:
    raise FileNotFoundError(f"No .tif files found in: {SEGMENTED_BATCH_DIR}")

current_index = 0
stats_list = []

# viewer einmal erstellen 
viewer = napari.Viewer(title="quality control images")


def load_new_case(viewer, image_path):
    img = iio.imread(image_path)  # (H, W, 3)

    layer1 = img[:, :, 0]               # original
    layer2 = img[:, :, 2].astype(np.int32)  # vessels
    layer3 = img[:, :, 1].astype(np.int32)  # region

    viewer.layers.clear()

    viewer.add_image(layer1, name="original_image", colormap="gray")
    viewer.add_labels(layer2, name="vessels", opacity=0.5)
    viewer.add_labels(layer3, name="region", opacity=0.5)

    
    v_layer = viewer.layers["vessels"]
    r_layer = viewer.layers["region"]
    # wichtig: direct mode erzwingen
    v_layer.color_mode = "direct"
    r_layer.color_mode = "direct"

# RGBA (0..1). Hintergrund transparent.
    v_layer.color = {0: (0, 0, 0, 0), 1: (0, 0, 1, 1)}      # blue
    r_layer.color = {0: (0, 0, 0, 0), 1: (1, 0.5, 0, 1)}    # orange

    v_layer.refresh()
    r_layer.refresh()
    
    viewer.status = f"{current_index+1} / {len(image_files)}   |   {Path(image_path).name}"


def show_current():
    load_new_case(viewer, image_files[current_index])


# initial load
show_current()


@viewer.bind_key('Right', overwrite=True)
def forward_image(viewer):
    global current_index
    if current_index < len(image_files) - 1:
        current_index += 1
        show_current()


@viewer.bind_key('Left', overwrite=True)
def backward_image(viewer):
    global current_index
    if current_index > 0:
        current_index -= 1
        show_current()


@viewer.bind_key('Up', overwrite=True)
def accept_image(viewer):
    viewer.status = f"GOOD  |  {current_index+1}/{len(image_files)}  |  {Path(image_files[current_index]).name}"
    print("GOOD:", image_files[current_index])


@viewer.bind_key('Down', overwrite=True)
def reject_image(viewer):
    viewer.status = f"BAD   |  {current_index+1}/{len(image_files)}  |  {Path(image_files[current_index]).name}"
    print("BAD:", image_files[current_index])

NameError: name 'layer2' is not defined

idee: cool wäre es wenn man einfach die message appended in eine liste, die dann in eine separate csv kommt und dann mit den pfaden der stats tabelle gematched werden kann 

In [ ]:
#brainstorming
empty_qc_list = []
fill list with [[path1, score1][path2,score2]] 
index fürs fillen == index aus glob listdir 
bad == 0, good == 1 
create csv -> f"{BATCH_DATE} + QC.csv"

csv.append_qc list 

